In [ ]:
# === Cellule 1 : Setup pour le run IMPROVED RSNA ===
from pathlib import Path
import sys
import json
import time
import pandas as pd
from tqdm import tqdm

# Path setup
ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.medgemma_inference import medgemma_predict
from src.guardrails import apply_safety_guardrails, validate_prediction
from src.database import insert_run, init_db

# ⚠️ IMPORTANT : on charge la MÊME sélection de 50 cas que pour baseline
# pour permettre une comparaison équitable (mêmes patients, même seed)
cases_full = pd.read_csv(ROOT / "data" / "rsna_png" / "rsna_cases.csv")

import random
random.seed(42)  # ← même seed que baseline pour reproductibilité
normal = cases_full[cases_full["label"] == "normal"].sample(n=25, random_state=42)
pneumo = cases_full[cases_full["label"] == "suspected_opacity"].sample(n=25, random_state=42)
cases_df = pd.concat([normal, pneumo]).reset_index(drop=True)

print(f"✅ {len(cases_df)} cas sélectionnés (MÊME échantillon que baseline)")
print(f"   Distribution : {dict(cases_df['label'].value_counts())}")

# SQLite + output dir
db_path = ROOT / "eval" / "medical_ai_evidence.sqlite"
init_db(db_path)
out_dir = ROOT / "eval" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

print(f"⏱️ ETA : 50 cas × ~3 min = ~2h30")
print(f"🆚 Comparaison avec baseline : SQLite et CSV de baseline déjà présents")
cases_df.head()